In [14]:
import subprocess
import time
import requests
import os
import socket

# --- CONFIGURATION ---
WARP_CLI_PATH = r"C:\Program Files\Cloudflare\Cloudflare WARP\warp-cli.exe"

# We will let the script find the correct port automatically
FOUND_PORT = None

def find_warp_port():
    """Scans for the Cloudflare WARP Proxy port."""
    # Common ports: 40000 is default for WARP, 1080 is standard SOCKS
    potential_ports = [40000, 1080, 8080, 9091]
    
    print("🔍 Scanning for WARP Proxy Port...")
    
    for port in potential_ports:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(0.5) # Fast check
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        
        if result == 0:
            print(f"✅ Found active service on port: {port}")
            return port
            
    return None

def get_public_ip(proxy_port):
    """Fetches IP through the specific proxy port."""
    proxy_url = f"socks5://127.0.0.1:{proxy_port}"
    proxies = {'http': proxy_url, 'https': proxy_url}
    try:
        # We use a specialized IP echo service
        return requests.get("https://api.ipify.org", proxies=proxies, timeout=5).text
    except Exception as e:
        return f"Error: {e}"

def connect_warp():
    if not os.path.exists(WARP_CLI_PATH):
        print(f"❌ Error: Could not find warp-cli.exe at: {WARP_CLI_PATH}")
        return

    print("--- 🔌 Connecting to Cloudflare WARP ---")
    
    # 1. Force Connect via CLI
    subprocess.run([WARP_CLI_PATH, "connect"], capture_output=True)
    
    # Give it time to handshake
    print("⏳ Waiting 5 seconds for connection...")
    time.sleep(5)

    # 2. Check Status
    status = subprocess.run([WARP_CLI_PATH, "status"], capture_output=True, text=True)
    if "Connected" in status.stdout:
        print("✅ WARP System Status: CONNECTED")
    else:
        print(f"⚠️ WARP Status: {status.stdout.strip()}")
        print("   (Attempting to proceed anyway via Proxy Port...)")

    # 3. Find the Port
    port = find_warp_port()
    if not port:
        print("❌ CRITICAL: Could not find an open Proxy Port.")
        print("   Please go to WARP App > Settings > Advanced > Configure Proxy.")
        print("   Make sure the port is set to 40000 or 1080.")
        return

    # 4. Verify IP Masking
    print(f"\n--- 🕵️ Testing Connection via Port {port} ---")
    try:
        warp_ip = get_public_ip(port)
        
        if "Error" in warp_ip:
            print(f"❌ Connection Failed: {warp_ip}")
        else:
            print(f"🛡️  SUCCESS! Your Script IP is hidden: {warp_ip}")
            print(f"\n👉 IMPORTANT: Update your scraper script to use port {port}")
            
    except Exception as e:
        print(f"❌ Proxy check failed: {e}")

if __name__ == "__main__":
    connect_warp()

--- 🔌 Connecting to Cloudflare WARP ---
⏳ Waiting 5 seconds for connection...
✅ WARP System Status: CONNECTED
🔍 Scanning for WARP Proxy Port...
✅ Found active service on port: 40000

--- 🕵️ Testing Connection via Port 40000 ---
🛡️  SUCCESS! Your Script IP is hidden: 104.28.193.166

👉 IMPORTANT: Update your scraper script to use port 40000


In [15]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import time
import json
import re
from datetime import datetime, timezone
from pathlib import Path

# --- PROXY CONFIGURATION ---
WARP_PROXY = "socks5://127.0.0.1:40000"
PROXIES = {
    'http': WARP_PROXY,
    'https': WARP_PROXY
}

def check_ip_safety():
    """Verifies that the proxy is working before we start scraping."""
    print("🛡️  Verifying Proxy Connection...")
    try:
        response = requests.get("https://api.ipify.org?format=json", proxies=PROXIES, timeout=10)
        proxy_ip = response.json().get('ip')
        print(f"✅ Secure Connection Established. Scraper IP: {proxy_ip}")
        return True
    except Exception as e:
        print(f"❌ CRITICAL ERROR: Could not connect through WARP Proxy.")
        print(f"   Error details: {e}")
        return False

def process_comment_tree(comment_data, thread_author, depth_limits, current_depth=0):
    """
    Recursively builds a tree.
    OPTIMIZATION 1: Injects [OP] and [MOD] tags for authority.
    OPTIMIZATION 2: Captures 'score' for quality filtering.
    """
    limit = depth_limits.get(current_depth, 0)
    if limit == 0: return []

    processed_comments = []
    # Fetch a few extra to account for filtering deleted comments
    children = comment_data.get('children', [])[:limit + 2] 
    
    count = 0
    for child in children:
        if count >= limit: break
        
        data = child.get('data', {})
        if child.get('kind') == 'more': continue
            
        body = data.get('body')
        author = data.get('author')
        distinguished = data.get('distinguished') # Check if Mod
        score = data.get('score', 0)
        
        if body and body not in ["[deleted]", "[removed]"]:
            
            # --- AUTHORITY INJECTION ---
            if distinguished == 'moderator':
                body = f"🛡️ [MODERATOR]: {body}"
            elif author == thread_author:
                body = f"🔴 [OP/CREATOR]: {body}" 
            
            comment_obj = {
                "body": body,
                "score": score,   # <--- CRITICAL FOR RAG FILTERING
                "replies": [] 
            }

            replies_raw = data.get('replies')
            if isinstance(replies_raw, dict):
                reply_tree = replies_raw.get('data', {})
                comment_obj['replies'] = process_comment_tree(
                    reply_tree, 
                    thread_author, 
                    depth_limits, 
                    current_depth + 1
                )
            
            processed_comments.append(comment_obj)
            count += 1
            
    return processed_comments

def get_reddit_data(
    subreddit, 
    output_folder, 
    post_limit=1000, 
    sort_by='top',
    time_filter='all',
    depth_limits=None 
):
    if not check_ip_safety(): return

    if depth_limits is None:
        depth_limits = {0: 25, 1: 15, 2: 10}

    target_path = Path(output_folder).resolve()
    target_path.mkdir(parents=True, exist_ok=True)

    session = requests.Session()
    session.proxies.update(PROXIES)
    retries = Retry(total=5, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
    session.mount('https://', HTTPAdapter(max_retries=retries))
    
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
    
    posts_collected = 0
    after = None
    
    sort_display = sort_by.upper()
    if sort_by == 'top': sort_display += f" OF {time_filter.upper()}"
    
    print(f"🚀 Scraping r/{subreddit} via WARP Proxy...")
    print(f"🎯 Target: {post_limit} posts (Sorted by {sort_display})")

    while posts_collected < post_limit:
        if sort_by == 'top':
            list_url = f"https://www.reddit.com/r/{subreddit}/top.json?t={time_filter}&limit=100"
        else:
            list_url = f"https://www.reddit.com/r/{subreddit}/{sort_by}.json?limit=100"
        
        if after: list_url += f"&after={after}"
            
        try:
            res = session.get(list_url, headers=headers, timeout=15).json()
            if 'data' not in res: break
            posts = res['data']['children']
            after = res['data']['after']
        except Exception as e:
            print(f"❌ Listing Error: {e}")
            break

        if not posts: break

        for post in posts:
            if posts_collected >= post_limit: break
            
            p_data = post['data']
            thread_id = p_data['id']
            thread_author = p_data.get('author')
            
            safe_title = re.sub(r'[<>:"/\\|?*]', '', p_data['title'])[:50].strip()
            filename = f"{thread_id}_{safe_title}.json"
            file_path = target_path / filename
            
            if file_path.exists():
                print(f"⏩ Skipping {filename}")
                posts_collected += 1
                continue

            thread_url = f"https://www.reddit.com/r/{subreddit}/comments/{thread_id}/.json?sort=top"
            
            try:
                thread_res = session.get(thread_url, headers=headers, timeout=15).json()
                if not isinstance(thread_res, list) or len(thread_res) < 2: continue

                structured_comments = process_comment_tree(
                    thread_res[1]['data'], 
                    thread_author=thread_author, 
                    depth_limits=depth_limits, 
                    current_depth=0
                )
                
                # --- OPTIMIZED DOCUMENT OBJECT ---
                doc_object = {
                    "meta": {
                        "title": p_data['title'],
                        "url": f"https://reddit.com{p_data['permalink']}",
                        "score": p_data.get('score', 0),
                        "flair": p_data.get('link_flair_text'), # <--- ADDED: Filter by Topic
                        "date": datetime.fromtimestamp(p_data.get('created_utc', 0), timezone.utc).strftime('%Y-%m-%d'),
                        "sort": sort_display
                    },
                    "content": {
                        "post_body": p_data.get('selftext', ''),
                        "comments": structured_comments
                    }
                }
                
                with open(file_path, "w", encoding="utf-8") as f:
                    json.dump(doc_object, f, indent=4, ensure_ascii=False)
                
                posts_collected += 1
                print(f"✅ [{posts_collected}/{post_limit}] Saved: {filename}")
                time.sleep(2) 
                
            except Exception as e:
                print(f"⚠️ Error on thread {thread_id}: {e}")

        if not after: break

    print(f"\n🎉 Extraction complete! Total posts collected: {posts_collected}")
    return posts_collected

if __name__ == "__main__":
    TARGET_FOLDER = r"D:\Nithin\Reddit\rag_ready_data"

    MY_DEPTH_LIMITS = {
        0: 25, 
        1: 15, 
        2: 10 
    }

    get_reddit_data(
        "ManchesterUnited", 
        output_folder=TARGET_FOLDER,
        post_limit=100,
        sort_by='top',      
        time_filter='year',
        depth_limits=MY_DEPTH_LIMITS 
    )

🛡️  Verifying Proxy Connection...
✅ Secure Connection Established. Scraper IP: 104.28.193.166
🚀 Scraping r/ManchesterUnited via WARP Proxy...
🎯 Target: 100 posts (Sorted by TOP OF YEAR)
✅ [1/100] Saved: 1ks8yvh_I've never been so embarrassed as a Man Utd fan.json
✅ [2/100] Saved: 1quhnjd_This is what ex-players should be doing instead of.json
✅ [3/100] Saved: 1n96kf1_Our Draco after joining the Death Eaters.json
✅ [4/100] Saved: 1qmr0t2_CUNHHAAAAAAAA.json
✅ [5/100] Saved: 1p0mocf_Scotland (1)-0 Denmark - Scott McTominay 3'.json
✅ [6/100] Saved: 1ngp35j_Napoli fan here, thanks again💙.json
✅ [7/100] Saved: 1n3lg03_Mocking that clip is WILD!.json
✅ [8/100] Saved: 1l05z77_This is the way back to the top..json


KeyboardInterrupt: 